In [5]:
import pandas as pd
df = pd.read_excel('O_SHAPE_DATA.xlsx')
df.head()

,WIDTH,LENGTH,HIGHT,ORIENTATION,FORM_FACTOR,S/V_RATIO,COOLING_LOAD,HEATING_LOAD,TOTAL_LOAD
0,30,60,16,135,0.9,0.225,48.776828,2.161788,50.938616
1,60,30,8,180,1.4,0.350,50.182264,0.734736,50.917001
2,60,30,8,0,1.4,0.350,50.182264,0.734736,50.917000
3,30,60,8,90,1.4,0.350,50.179383,0.736617,50.916000
4,30,60,8,270,1.4,0.350,50.179383,0.736617,50.916000


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
import pandas as pd


df['Energy_Class'] = pd.qcut(df['TOTAL_LOAD'], q=3, labels=['Low', 'Medium', 'High'])


X = df.drop(['COOLING_LOAD', 'HEATING_LOAD', 'TOTAL_LOAD', 'Energy_Class'], axis=1)
y = df['Energy_Class']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

print(f"Random Forest Accuracy (O Shape): {accuracy_score(y_test, rf_preds) * 100:.2f}%")
print(f"Random Forest F1-Score (O Shape): {f1_score(y_test, rf_preds, average='weighted') * 100:.2f}%\n")

le = LabelEncoder()
y_train_xgb = le.fit_transform(y_train)
y_test_xgb = le.transform(y_test)

xgb_model = XGBClassifier(random_state=42)
xgb_model.fit(X_train, y_train_xgb)
xgb_preds = xgb_model.predict(X_test)

print(f"XGBoost Accuracy (O Shape): {accuracy_score(y_test_xgb, xgb_preds) * 100:.2f}%")
print(f"XGBoost F1-Score (O Shape): {f1_score(y_test_xgb, xgb_preds, average='weighted') * 100:.2f}%")

Random Forest Accuracy (O Shape): 35.29%
Random Forest F1-Score (O Shape): 34.13%

XGBoost Accuracy (O Shape): 36.27%
XGBoost F1-Score (O Shape): 35.20%


In [7]:
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, Conv1D, Flatten, Input
import numpy as np
from sklearn.metrics import accuracy_score, f1_score


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_aug, y_train_aug = smote.fit_resample(X_train_scaled, y_train_xgb)

print("Training Deep Learning Models for O Shape, please wait...\n")

ann_model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_aug.shape[1],)),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])
ann_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
ann_model.fit(X_train_aug, y_train_aug, epochs=50, batch_size=32, validation_data=(X_test_scaled, y_test_xgb), verbose=0)
ann_preds = np.argmax(ann_model.predict(X_test_scaled, verbose=0), axis=1)
print(f"ANN Accuracy (O Shape): {accuracy_score(y_test_xgb, ann_preds) * 100:.2f}%")
print(f"ANN F1-Score (O Shape): {f1_score(y_test_xgb, ann_preds, average='weighted') * 100:.2f}%\n")


X_train_rnn = X_train_aug.reshape((X_train_aug.shape[0], 1, X_train_aug.shape[1]))
X_test_rnn = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

rnn_model = Sequential([
    Input(shape=(1, X_train_aug.shape[1])),
    SimpleRNN(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(3, activation='softmax')
])
rnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
rnn_model.fit(X_train_rnn, y_train_aug, epochs=50, batch_size=32, validation_data=(X_test_rnn, y_test_xgb), verbose=0)
rnn_preds = np.argmax(rnn_model.predict(X_test_rnn, verbose=0), axis=1)
print(f"RNN Accuracy (O Shape): {accuracy_score(y_test_xgb, rnn_preds) * 100:.2f}%")
print(f"RNN F1-Score (O Shape): {f1_score(y_test_xgb, rnn_preds, average='weighted') * 100:.2f}%\n")


cnn_model = Sequential([
    Input(shape=(1, X_train_aug.shape[1])),
    Conv1D(filters=32, kernel_size=1, activation='relu'),
    Flatten(),
    Dense(16, activation='relu'),
    Dense(3, activation='softmax')
])
cnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn_model.fit(X_train_rnn, y_train_aug, epochs=50, batch_size=32, validation_data=(X_test_rnn, y_test_xgb), verbose=0)
cnn_preds = np.argmax(cnn_model.predict(X_test_rnn, verbose=0), axis=1)
print(f"CNN Accuracy (O Shape): {accuracy_score(y_test_xgb, cnn_preds) * 100:.2f}%")
print(f"CNN F1-Score (O Shape): {f1_score(y_test_xgb, cnn_preds, average='weighted') * 100:.2f}%")

Training Deep Learning Models for O Shape, please wait...



/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


ANN Accuracy (O Shape): 46.26%
ANN F1-Score (O Shape): 44.26%

RNN Accuracy (O Shape): 46.17%
RNN F1-Score (O Shape): 44.74%

CNN Accuracy (O Shape): 45.81%
CNN F1-Score (O Shape): 44.19%
